### Recreate Feature Matrix

In [ ]:
import polars as pl
from sklearn.preprocessing import StandardScaler

%store -r df_north
%store -r df_south

# join household metadata
df_north_combined = pl.concat(
    [df_north["household"], df_north["water_usage"]],
    how="horizontal"
)
df_south_combined = pl.concat(
    [df_south["household"], df_south["water_usage"]],
    how="horizontal"
)
# extract household ids
north_household_ids = df_north_combined["household_id"]
south_household_ids = df_south_combined["household_id"]
# drop household id column
north_features = df_north_combined.drop("household_id")
south_features = df_south_combined.drop("household_id")
# encode landscape types
north_features = north_features.to_dummies(columns=["landscape_type"])
south_features = south_features.to_dummies(columns=["landscape_type"])
# convert to numpy
X_raw_north = north_features.to_numpy()
X_raw_south = south_features.to_numpy()
# scale
scaler = StandardScaler()
X_scaled_north = scaler.fit_transform(X_raw_north)
X_scaled_south = scaler.fit_transform(X_raw_south)

### Cluster Separability Function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

path = Path('/content/drive/MyDrive/hydromind')
fig_path = path / "figures"
fig_path.mkdir(parents=True, exist_ok=True)

def cluster_suitability(feature_matrix: np.ndarray, name: str, max_k: int = 8) -> dict[str, list]:
    # PCA / t-SNE visualization + Elbow + Silhoutte Analysis
    pca = PCA(n_components=min(15, feature_matrix.shape[1]))
    pca_result = pca.fit_transform(feature_matrix)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # scree plot
    axes[0].bar(
        range(1, len(pca.explained_variance_ratio_) + 1),
        pca.explained_variance_ratio_,
        color="teal",
        alpha=0.7
    )
    axes[0].set_title("PCA Scree Plot")
    axes[0].set_xlabel("Principal Component")
    axes[0].set_ylabel("Explained Variance Ratio")

    # t-SNE 2D projection
    tsne_sample_size = min(5000, feature_matrix.shape[0])
    random_indices = np.random.choice(feature_matrix.shape[0], tsne_sample_size, replace=False)
    X_tsne_sample = feature_matrix[random_indices]

    tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
    tsne_result = tsne.fit_transform(X_tsne_sample)

    axes[1].scatter(tsne_result[:, 0], tsne_result[:, 1], s=5, alpha=0.6, c="steelblue")
    axes[1].set_title(f"t-SNE 2D Projection (n={tsne_sample_size})")
    axes[1].axis('off')

    # elbow + silhouette
    inertias, silhouettes = [], []
    for k in range(2, max_k + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = km.fit_predict(feature_matrix)
        inertias.append(km.inertia_)

        score = silhouette_score(feature_matrix, labels, sample_size=2000, random_state=42)
        silhouettes.append(score)

    ax2 = axes[2]
    ax2.plot(range(2, max_k + 1), inertias, "o-", color="crimson", label="Inertia")
    ax2.set_ylabel("Inertia", color="crimson")

    ax2b = ax2.twinx()
    ax2b.plot(range(2, max_k + 1), silhouettes, "s-", color="teal", label="Silhouette")
    ax2b.set_ylabel("Silhouette Score", color="teal")

    ax2.set_xlabel("Number of Clusters (k)")
    ax2.set_title("Elbow + Silhouette Analysis")

    # combine legends
    lines_1, labels_1 = ax2.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')

    plt.tight_layout()
    plt.savefig(f"{fig_path}/Cluster Analysis ({name}).png", dpi=150)
    plt.close()

    return {
        "name": name,
        "pca_variance": pca.explained_variance_ratio_.tolist(), 
        "silhouettes": silhouettes
    }

north_results = cluster_suitability(X_scaled_north, name="North Hemisphere",max_k=8)
south_results = cluster_suitability(X_scaled_south, name="South Hemisphere",max_k=8)
print(north_results)
print(south_results)